# Butly LoCoMo Evaluation (Colab Pro)

This notebook is a **thin frontend**: it mounts Drive, prepares the repo and
local model servers, then drives `python -m evals.locomo.cli`. All evaluation
logic lives in `evals/locomo/` — do not add scoring, replay, or checkpoint
code here.

**Role-based model servers.** Each Butly role (chat / gatekeeper / summary /
knowledge / embedding) can use its own model, configured in the Parameters
cell. Roles that share the same model + port share one server. Reasoning
(thinking) models are accurate but slow; assign Non-Reasoning models to
gatekeeper / summary / knowledge for practical throughput. Embeddings need a
dedicated embeddings-capable server (a chat LLM cannot answer
`/v1/embeddings`).

Prerequisites:

* Use a **GPU runtime** (Runtime -> Change runtime type -> GPU; L4 is fine).
* Put the LoCoMo dataset JSON on Drive. Official data is CC BY-NC 4.0 and is
  **not** bundled with Butly — download it from
  https://github.com/snap-research/locomo yourself.
* Optionally add `HF_TOKEN` to Colab Secrets for gated/rate-limited downloads.

llama.cpp is built fresh each session (a few minutes); Drive binary caching
was removed after repeated shared-library breakage. Artifacts (checkpoints
included) are written to Drive, so a disconnected runtime can continue with
the **Resume** cell near the end.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Parameters (edit these) ---
REPO_URL = 'https://github.com/unagisann/Butly.git'
BRANCH = 'main'
REPO_DIR = '/content/butly'

DRIVE_ROOT = '/content/drive/MyDrive/butly-evals'
DATASET_PATH = f'{DRIVE_ROOT}/data/locomo10.json'
RUN_ID = 'qwen3_14b_colab_v7'   # one run directory per model + attempt

# --- Model roles ---
# Roles sharing the same hf_repo/hf_file/port share one server process.
# chat is the model under evaluation (Reasoning OK). gatekeeper/summary/
# knowledge run many times per session, so a Non-Reasoning model keeps the
# run practical. VRAM guide for L4 (22.5GB): Qwen3-14B Q4 ~9GB +
# Gemma4-12B Q4 ~7GB + embedding ~0.1GB + KV caches fits.
MODELS = {
    'chat': dict(
        hf_repo='Qwen/Qwen3-14B-GGUF',
        hf_file='Qwen3-14B-Q4_K_M.gguf',
        model_name='qwen3-14b',
        port=8090,
    ),
    'gatekeeper': dict(
        hf_repo='unsloth/gemma-4-12b-it-GGUF',
        hf_file='gemma-4-12b-it-Q4_K_M.gguf',
        model_name='gemma-4-12b-it',
        port=8092,
    ),
    'summary': dict(
        hf_repo='unsloth/gemma-4-12b-it-GGUF',
        hf_file='gemma-4-12b-it-Q4_K_M.gguf',
        model_name='gemma-4-12b-it',
        port=8092,
    ),
    'knowledge': dict(
        hf_repo='unsloth/gemma-4-12b-it-GGUF',
        hf_file='gemma-4-12b-it-Q4_K_M.gguf',
        model_name='gemma-4-12b-it',
        port=8092,
    ),
    'embedding': dict(
        hf_repo='nomic-ai/nomic-embed-text-v1.5-GGUF',
        hf_file='nomic-embed-text-v1.5.Q4_K_M.gguf',
        model_name='nomic-embed-text',
        port=8091,
        embeddings=True,
    ),
}

SAMPLE_LIMIT = 1
SESSION_LIMIT = 3
QUESTION_LIMIT = 10

# 1 server per port; reject conflicting assignments
server_specs = {}
for role, spec in MODELS.items():
    existing = server_specs.get(spec['port'])
    if existing and (existing['hf_repo'], existing['hf_file']) != (spec['hf_repo'], spec['hf_file']):
        raise ValueError(f"port {spec['port']} is assigned two different models")
    merged = dict(existing or {})
    merged.update(spec)
    server_specs[spec['port']] = merged
print('servers:', {port: s['model_name'] for port, s in sorted(server_specs.items())})


In [ ]:
# --- Clone / update Butly and install dependencies ---
import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# --- Build llama.cpp (fresh each session, run in place from build/bin) ---
# No Drive caching: cached binaries repeatedly broke on missing shared
# libraries. Running from build/bin keeps the RPATH valid.
import os
!apt-get -qq install -y libcurl4-openssl-dev > /dev/null
![ -d /content/llama.cpp ] || git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-server -j > /dev/null
SERVER_BIN = '/content/llama.cpp/build/bin/llama-server'
print('binary exists:', os.path.isfile(SERVER_BIN))


In [ ]:
# --- Download the GGUF models for every server ---
import os
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
except Exception:
    pass  # token only needed for gated / rate-limited downloads

!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
for port, spec in sorted(server_specs.items()):
    spec['model_path'] = hf_hub_download(spec['hf_repo'], spec['hf_file'])
    print(port, spec['model_name'], '->', spec['model_path'])


In [ ]:
# --- Robust server launcher: kill stale server on the port, log to file,
#     wait for /health, surface the log on failure ---
import subprocess, time, socket, pathlib, urllib.request

def _port_free(port):
    s = socket.socket()
    try:
        s.bind(('127.0.0.1', port)); return True
    except OSError:
        return False
    finally:
        s.close()

def start_llama_server(model_path, port, extra_args=None, timeout=600):
    extra_args = extra_args or []
    # free the port (kill a previous llama-server bound to it)
    subprocess.run(['pkill', '-9', '-f', f'--port {port}'], capture_output=True)
    for _ in range(15):
        if _port_free(port):
            break
        time.sleep(2)
    else:
        raise RuntimeError(f'port {port} is busy and did not free up; pick another in Parameters')

    log_path = f'/content/llama_{port}.log'
    log = open(log_path, 'w')
    proc = subprocess.Popen(
        [SERVER_BIN, '-m', model_path, '--port', str(port), '-ngl', '99'] + extra_args,
        stdout=log, stderr=subprocess.STDOUT,
    )
    for i in range(timeout // 2):
        if proc.poll() is not None:
            print(f'server on {port} EXITED with code', proc.returncode)
            print(pathlib.Path(log_path).read_text()[-3000:])
            raise RuntimeError(f'llama-server on {port} exited; see log above')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=2)
            print(f'server on {port} is UP')
            return proc
        except Exception:
            if i % 15 == 14:
                tail = pathlib.Path(log_path).read_text().splitlines()
                print(f'  {port} loading...', tail[-1] if tail else '(no output yet)')
            time.sleep(2)
    raise RuntimeError(f'server on {port} did not become healthy in {timeout}s (see /content/llama_{port}.log)')

server_procs = {}
for port, spec in sorted(server_specs.items()):
    extra = ['--embeddings', '--pooling', 'mean'] if spec.get('embeddings') else []
    server_procs[port] = start_llama_server(spec['model_path'], port, extra_args=extra)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# --- Register each server as a Butly connection + write the role profile ---
import json, os, pathlib, urllib.request
import yaml

user_config = {
    'LLM_CONNECTIONS': [
        {
            'id': f'colab_{port}',
            'protocol': 'openai_compat',
            'base_url': f'http://127.0.0.1:{port}/v1',
            'api_key_env': 'COLAB_LOCAL_API_KEY',
            'label': f"Colab {spec['model_name']} (:{port})",
        }
        for port, spec in sorted(server_specs.items())
    ]
}
pathlib.Path('user_config.json').write_text(json.dumps(user_config, indent=2))
os.environ['COLAB_LOCAL_API_KEY'] = 'local'  # llama.cpp accepts any key

profile = {'name': 'colab_roles'}
for role, spec in MODELS.items():
    profile[role] = {
        'connection': f"colab_{spec['port']}",
        'model_name': spec['model_name'],
    }
pathlib.Path('evals/locomo/profiles/colab_roles.yaml').write_text(
    yaml.safe_dump(profile, sort_keys=False)
)
print(yaml.safe_dump(profile, sort_keys=False))

# sanity check: every server answers /health
for port in sorted(server_specs):
    body = urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=3).read().decode()
    print(f':{port} /health:', body)
print('connections + profile ready')


In [ ]:
# --- Run the evaluation (replay -> sleeptime -> QA -> score -> report) ---
!python -m evals.locomo.cli run \
  --dataset "{DATASET_PATH}" \
  --output-dir "{DRIVE_ROOT}/runs" \
  --run-id "{RUN_ID}" \
  --profile evals/locomo/profiles/colab_roles.yaml \
  --sample-limit {SAMPLE_LIMIT} \
  --session-limit {SESSION_LIMIT} \
  --question-limit {QUESTION_LIMIT}


In [ ]:
# --- Resume after a runtime disconnect (safe to re-run; skips finished work) ---
# Re-run the setup cells above first (mount, clone, build, download, servers,
# connections), then run this cell instead of the run cell.
!python -m evals.locomo.cli resume --run-dir "{DRIVE_ROOT}/runs/{RUN_ID}"

In [ ]:
# --- Show the summary ---
from IPython.display import Markdown, display
import pathlib
display(Markdown(pathlib.Path(f'{DRIVE_ROOT}/runs/{RUN_ID}/summary.md').read_text()))